In [ ]:
from pathlib import Path
import json

# Find the root directory, no matter where we are. 
def find_project_root(marker="README.md"):
    p = Path.cwd()
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise RuntimeError("Project root not found")

PROJECT_ROOT = find_project_root()

path = PROJECT_ROOT / "notebooks/api_progress.json"

In [ ]:
# Read in feature engineered data 

input_path = PROJECT_ROOT / "data/processed/feature_engineered_data.csv"
import pandas as pd
df = pd.read_csv(input_path)

In [ ]:
df['title'] = df['title'].fillna('')
df['description'] = df['description'].fillna('')

In [ ]:
df['title_description'] = df['title'] + ' ' + df['description']

In [ ]:
df['missing_long_lat'] = df['longitude'].isna() | df['latitude'].isna()

In [ ]:
company_counts = df['company.display_name'].value_counts()
df['company_listing_count'] = df['company.display_name'].map(company_counts)

In [ ]:
min_count = 50  # tune this threshold

for col in ['location_region', 'location_city']:
    counts = df[col].value_counts()
    df[col] = df[col].where(df[col].map(counts) >= min_count, 'Other')

In [ ]:
# Remove rows with salaries below 50k and cap salaries above 300k. We've found that there's a lot of data entry errors (like 20 and 40) below the 50k mark. 

lower = 50_000
upper = 300_000

df = df[df['avg_salary'] >= lower].copy()
df['avg_salary'] = df['avg_salary'].clip(upper=upper)

In [ ]:
experiments = [
    {'text': 'title',    'features': 'tfidf',       'reduce': False},
    {'text': 'title_description',      'features': 'tfidf',       'reduce': True},
    {'text': 'title',    'features': 'embeddings',  'reduce': False},
    {'text': 'title',    'features': 'embeddings',  'reduce': True},
    {'text': 'title_description',      'features': 'embeddings',  'reduce': False},
    {'text': 'title_description',      'features': 'tfidf',  'reduce': False}# ,
    # Removed this as computationally expensive to run (takes more than 5 minutes per fold) 
    # and we already know that reduce on description + no reduce on title is better than reduce on both. 
    # {'text': 'title_description',      'features': 'both',        'reduce': True},
    # add others if early results suggest they're interesting
]

In [ ]:
pd.DataFrame(experiments)

In [ ]:
X_structured = df.drop(columns=['avg_salary', 'title', 'description'])
X_structured.columns

cat_vars = ['contract_type', 'contract_time', 'category.label', 'location.area_length', 
            # 'location_country', 
            'location_state', 'location_region', 'location_city', 'missing_long_lat']

num_vars = ['longitude', 'latitude', 'company_listing_count']

In [ ]:
# df[num_vars].apply(lambda x: type(x[0]))

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import numpy as np
from sklearn.impute import SimpleImputer

df = df.reset_index(drop=True)
y = df['avg_salary'].values

# encode categoricals
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
cat_encoded = ohe.fit_transform(df[cat_vars])


imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()

# scale continuous
cont_imputed = imputer.fit_transform(df[num_vars])
cont_scaled = scaler.fit_transform(cont_imputed)


# combine into one non-text feature matrix
X_structured = np.hstack([cat_encoded, cont_scaled])

In [ ]:
nan_cols = np.where(np.isnan(X_structured).any(axis=0))[0]
print(nan_cols)

In [ ]:
# Model training and validation functions 

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sentence_transformers import SentenceTransformer
import pandas as pd

EMBEDDING_MODEL = SentenceTransformer('all-MiniLM-L6-v2')


def get_text_series(df, text_config):
    """
    Select the appropriate text column from the dataframe.
    text_config: 'title', 'description', or 'title_description'
    """
    valid = ['title', 'description', 'title_description']
    if text_config not in valid:
        raise ValueError(f"text_config must be one of {valid}, got '{text_config}'")
    return df[text_config]


def build_text_features(train_series, val_series, feature_config, reduce=False, n_components=100):
    """
    Fit text transformers on train_series, transform both train and val.
    feature_config: 'tfidf', 'embeddings', or 'both'
    Returns X_train, X_val as numpy arrays.
    """

    def get_tfidf(train, val):
        tfidf = TfidfVectorizer(max_features=10_000, ngram_range=(1, 2), min_df=3, sublinear_tf=True)
        X_train = tfidf.fit_transform(train)
        X_val = tfidf.transform(val)
        if reduce:
            svd = TruncatedSVD(n_components=n_components, random_state=42)
            X_train = svd.fit_transform(X_train)
            X_val = svd.transform(X_val)
        else:
            X_train = X_train.toarray()
            X_val = X_val.toarray()
        return X_train, X_val

    def get_embeddings(train, val):
        X_train = EMBEDDING_MODEL.encode(train.tolist(), batch_size=64, show_progress_bar=False)
        X_val = EMBEDDING_MODEL.encode(val.tolist(), batch_size=64, show_progress_bar=False)
        if reduce:
            pca = PCA(n_components=n_components, random_state=42)
            X_train = pca.fit_transform(X_train)
            X_val = pca.transform(X_val)
        return X_train, X_val

    if feature_config == 'tfidf':
        return get_tfidf(train_series, val_series)
    elif feature_config == 'embeddings':
        return get_embeddings(train_series, val_series)
    elif feature_config == 'both':
        tfidf_train, tfidf_val = get_tfidf(train_series, val_series)
        emb_train, emb_val = get_embeddings(train_series, val_series)
        return np.hstack([tfidf_train, emb_train]), np.hstack([tfidf_val, emb_val])
    else:
        raise ValueError(f"feature_config must be 'tfidf', 'embeddings', or 'both', got '{feature_config}'")


def evaluate(y_true, y_pred):
    """
    Calculate regression metrics.
    Returns dict with rmse, mae, r2.
    """
    return {
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'mae': mean_absolute_error(y_true, y_pred),
        'r2': r2_score(y_true, y_pred)
    }


def run_experiments(df, X_structured, y, experiments, use = 'both', n_splits=5, model_type='ridge'):
    """
    Run each experiment config across k folds, average metrics, return results dataframe.
    df: full dataframe containing text columns
    X_structured: pre-encoded non-text features as numpy array (n_samples, n_features)
    y: target array (n_samples,)
    experiments: list of dicts with keys 'text', 'features', 'reduce'
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    results = []

    for exp in experiments:
        print(f"Running: {exp}")
        fold_metrics = []

        for fold_num, (train_idx, val_idx) in enumerate(kf.split(df)):
            # Print statement so we can track progress more granularly
            print(f"  Fold {fold_num + 1}/{n_splits}")
            # slice text
            text_series = get_text_series(df, exp['text'])
            train_text = text_series.iloc[train_idx]
            val_text = text_series.iloc[val_idx]

            # build text features
            X_text_train, X_text_val = build_text_features(
                train_text, val_text,
                feature_config=exp['features'],
                reduce=exp['reduce']
            )

            # slice and combine with non-text features
            
            if use == 'text':
                X_train = X_text_train
                X_val = X_text_val
            elif use == 'structured':
                X_train = X_structured[train_idx]
                X_val = X_structured[val_idx]
            elif use == 'both':
                X_train = np.hstack([X_text_train, X_structured[train_idx]])
                X_val = np.hstack([X_text_val, X_structured[val_idx]])
            else:
                raise ValueError(f"use must be 'text', 'structured', or 'both', got '{use}'")

            y_train = y[train_idx]
            y_val = y[val_idx]

            # fit model
            model = Ridge(alpha=1.0) if model_type == 'ridge' else Lasso(alpha=0.1)
            model.fit(X_train, y_train)

            # evaluate
            y_pred = model.predict(X_val)
            fold_metrics.append(evaluate(y_val, y_pred))

        # average across folds
        avg_metrics = {k: np.mean([f[k] for f in fold_metrics]) for k in fold_metrics[0]}
        results.append({**exp, **avg_metrics})

    return pd.DataFrame(results)

In [ ]:
results = run_experiments(df, X_structured, y, experiments, n_splits=5, model_type='ridge')

In [ ]:
experiments[0]

In [ ]:
results_text = run_experiments(df, X_structured, y, experiments, use='text', n_splits=5, model_type='ridge')
results_structured = run_experiments(df, X_structured, y, [experiments[0]], use='structured', n_splits=5, model_type='ridge')

In [ ]:
# Plot RSME as a function of text, features, and reduce 
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
plt.figure(figsize=(12, 6))
# sns.barplot(data=results, x='text', y='rmse', hue='features')
# plt.title('RMSE by Text Configuration and Feature Type')
# plt.ylabel('RMSE')
# plt.xlabel('Text Configuration')
# plt.legend(title='Feature Type')
# Facet by reduce 
g = sns.catplot(data=results, x='text', y='mae', hue='features', col='reduce', kind='bar', height=5, aspect=1)
g.set_axis_labels("Text Configuration", "MAE")
g.set_titles("Dimensionality Reduction: {col_name}")
plt.show()

In [ ]:
# Plot predictive power of a text only model
sns.set_style('whitegrid')
plt.figure(figsize=(12, 6))
g = sns.catplot(data=results_text, x='text', y='mae', hue='reduce', col='reduce', kind='bar', height=5, aspect=1)
g.set_axis_labels("Text Configuration", "MAE")
g.set_titles("Dimensionality Reduction: {col_name}")
plt.show()

In [ ]:
# Get the median salary from out data 
median_salary = df['avg_salary'].median()
print(median_salary)

In [ ]:
df['title'].str.lower().value_counts()

In [ ]:
# Plot the salary distribution jsut for 'Software Engineer' 
plt.figure(figsize=(10, 5))
sns.histplot(df[df['title'].str.lower() == 'software engineer']['avg_salary'], bins=30, kde=True)
plt.title('Salary Distribution for Software Engineers')
plt.xlabel('Average Salary')
plt.ylabel('Frequency')
plt.show()

In [ ]:
results_text

In [ ]:
# Text the performance of a mdoel that reduces description but not title: 
n_splits = 5
use = 'text'
model_type = 'ridge'

kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
results = []


fold_metrics = []

for fold_num, (train_idx, val_idx) in enumerate(kf.split(df)):
    # Print statement so we can track progress more granularly
    print(f"  Fold {fold_num + 1}/{n_splits}")
    # slice text
    title_series = get_text_series(df, 'title')
    description_series = get_text_series(df, 'description')
    
    train_title_text = title_series.iloc[train_idx]
    val_title_text = title_series.iloc[val_idx]
    train_description_text = description_series.iloc[train_idx]
    val_description_text = description_series.iloc[val_idx]

    # build text features
    X_title_train, X_title_val = build_text_features(
        train_title_text, val_title_text,
        feature_config='tfidf',
        reduce= False 
    )
    
    X_description_train, X_description_val = build_text_features(
        train_description_text, val_description_text,
        feature_config='tfidf',
        reduce= True
    )

    # slice and combine with non-text features
    
    if use == 'text':
        X_train = np.hstack([X_title_train, X_description_train])
        X_val = np.hstack([X_title_val, X_description_val])
    elif use == 'structured':
        X_train = X_structured[train_idx]
        X_val = X_structured[val_idx]
    elif use == 'both':
        X_train = np.hstack([X_title_train, X_description_train, X_structured[train_idx]])
        X_val = np.hstack([X_title_val, X_description_val, X_structured[val_idx]])
    else:
        raise ValueError(f"use must be 'text', 'structured', or 'both', got '{use}'")

    y_train = y[train_idx]
    y_val = y[val_idx]

    # fit model
    model = Ridge(alpha=1.0) if model_type == 'ridge' else Lasso(alpha=0.1)
    model.fit(X_train, y_train)

    # evaluate
    y_pred = model.predict(X_val)
    fold_metrics.append(evaluate(y_val, y_pred))

print(fold_metrics)

In [ ]:
# get feature names
ohe_feature_names = ohe.get_feature_names_out(cat_vars)
feature_names = list(ohe_feature_names) + num_vars

# retrain on full dataset
model = Ridge(alpha=1.0)
model.fit(X_structured, y)

# build importance dataframe
importances = pd.DataFrame({
    'feature': feature_names,
    'coefficient': model.coef_
}).sort_values('coefficient', key=abs, ascending=False)

print(importances.head(30))